# This downloads Swiss locations for Migros, Coop, Aldi, Lidl and Denner from OpenStreetMap and saves both raw JSON and cleaned CSV files.

This file does:
- Downloads OpenStreetMap data
- Saves the original JSON
- Converts JSON into a DataFrame
- Inspects missing values and store names
- Removes duplicates
- makes store groups
- Removes records without coordinates
- Saves the cleaned CSV

In [24]:
import json
import requests
import pandas as pd

In [25]:
#sends a request to the OpenStreetMap Overpass API and asks:
#Find shops anywhere in Switzerland whose name or brand contains Migros, Migrolino, VOI, Coop, Denner, Lidl or Aldi.

# API address
overpass_url = "https://overpass-api.de/api/interpreter"

# node - gives a point with lattitudes and logitudes (ex. a point)
# way - connects multiple nodes (roads, paths, buildings, store boundaries) (ex. a outline of the store) -"out center" to give single point
# relation - groups multiple nodes, ways or others into one object (ex. shopping complex)
overpass_query = """
[out:json][timeout:180];

area["ISO3166-1"="CH"]["boundary"="administrative"]->.switzerland;

(
    node["shop"]["name"~"Migros|Migrolino|VOI|Coop|Denner|Lidl|Aldi", i]
        (area.switzerland);

    way["shop"]["name"~"Migros|Migrolino|VOI|Coop|Denner|Lidl|Aldi", i]
        (area.switzerland);

    relation["shop"]["name"~"Migros|Migrolino|VOI|Coop|Denner|Lidl|Aldi", i]
        (area.switzerland);

    node["shop"]["brand"~"Migros|Migrolino|VOI|Coop|Denner|Lidl|Aldi", i]
        (area.switzerland);

    way["shop"]["brand"~"Migros|Migrolino|VOI|Coop|Denner|Lidl|Aldi", i]
        (area.switzerland);

    relation["shop"]["brand"~"Migros|Migrolino|VOI|Coop|Denner|Lidl|Aldi", i]
        (area.switzerland);
);

out center tags;
"""

headers = {
    "User-Agent": "MigrosStoreLocationStudentProject/1.0"
}

response = requests.post(
    overpass_url,
    data={"data": overpass_query},
    headers=headers,
    timeout=240
)

response.raise_for_status()

store_data = response.json()

print("API request successful")
print("Number of records downloaded:", len(store_data["elements"]))

API request successful
Number of records downloaded: 3781


In [26]:
# Save the original JSON
with open(
    "data/switzerland_supermarkets_raw.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        store_data,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Raw JSON saved")

Raw JSON saved


In [27]:
# convert the API results into dataframe
store_records = []

for element in store_data["elements"]:

    tags = element.get("tags", {})

    # Nodes contain coordinates directly.
    # Ways and relations contain coordinates inside "center".
    latitude = element.get("lat")
    longitude = element.get("lon")

    if latitude is None:
        latitude = element.get("center", {}).get("lat")

    if longitude is None:
        longitude = element.get("center", {}).get("lon")

    store_records.append({
        "osm_type": element.get("type"),
        "osm_id": element.get("id"),
        "name": tags.get("name"),
        "brand": tags.get("brand"),
        "operator": tags.get("operator"),
        "shop_type": tags.get("shop"),
        "street": tags.get("addr:street"),
        "house_number": tags.get("addr:housenumber"),
        "postal_code": tags.get("addr:postcode"),
        "city": tags.get("addr:city"),
        "latitude": latitude,
        "longitude": longitude,
        "opening_hours": tags.get("opening_hours"),
        "website": tags.get("website")
    })

stores_df = pd.DataFrame(store_records)

print("Dataframe shape:", stores_df.shape)

stores_df.head()

Dataframe shape: (3781, 14)


,osm_type,osm_id,name,brand,operator,shop_type,street,house_number,postal_code,city,latitude,longitude,opening_hours,website
0,node,36726161,Migros,Migros,Genossenschaft Migros Zürich,supermarket,Wiesentalstrasse,4,8730,Uznach,47.228183,8.965330,Mo-Th 07:30-19:00; Fr 07:30-20:00; Sa 07:30-17...,https://filialen.migros.ch/de/migros-supermark...
1,node,39768209,Coop,Coop,Coop Genossenschaft,supermarket,NaN,NaN,8730,Uznach,47.225154,8.969868,NaN,NaN
2,node,39947904,Coop,Coop,Coop Genossenschaft,supermarket,Bahnhofbrücke,1,8001,Zürich,47.376732,8.542161,Mo-Sa 06:00-22:00,NaN
3,node,47744371,Coop Tankstelle Höngg,Coop,Coop Mineraloel,convenience,Am Wasser,146,8049,Zürich,47.399526,8.497329,Mo-Su 06:00-23:00,NaN
4,node,48932835,Migros,Migros,Genossenschaft Migros Zürich,supermarket,Wengistrasse,7,8004,Zürich,47.375020,8.522895,Mo-Sa 08:00-21:00; PH off,https://filialen.migros.ch/de/migros-supermark...


In [28]:
# inspect the downloaded data
stores_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3781 entries, 0 to 3780
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   osm_type       3781 non-null   str    
 1   osm_id         3781 non-null   int64  
 2   name           3777 non-null   str    
 3   brand          3340 non-null   str    
 4   operator       1551 non-null   str    
 5   shop_type      3781 non-null   str    
 6   street         2359 non-null   str    
 7   house_number   2309 non-null   str    
 8   postal_code    2256 non-null   str    
 9   city           2239 non-null   str    
 10  latitude       3781 non-null   float64
 11  longitude      3781 non-null   float64
 12  opening_hours  2960 non-null   str    
 13  website        1616 non-null   str    
dtypes: float64(2), int64(1), str(11)
memory usage: 413.7 KB


In [29]:
stores_df.isna().sum()

osm_type            0
osm_id              0
name                4
brand             441
operator         2230
shop_type           0
street           1422
house_number     1472
postal_code      1525
city             1542
latitude            0
longitude           0
opening_hours     821
website          2165
dtype: int64

In [30]:
stores_df["name"].value_counts().head(30)

name
Coop                           977
Denner                         702
Migros                         678
Migrolino                      262
Coop Pronto                    245
Lidl                           192
Aldi                           157
ALDI                            75
VOI                             69
Denner Express                  69
Migros Florissimo               49
Coop City                       23
Migros Partner                  17
Denner Satellit                 17
Coop to go                      10
M-Outlet                         9
Migros teo                       8
Denner Bibite                    7
Denner Partner                   7
Voi                              5
Denner Satellite                 5
Boulangerie Duvoisin             4
Coop Blumen                      4
Migros Outlet                    4
Aldi Suisse                      4
ALDI Suisse                      3
Migros Daily                     3
Migros Blumen                    3
Migros MMM Zent

In [31]:
# Remove exact duplicate OSM records
# Some shops may match both the name query and the brand query.
stores_df = (
    stores_df
    .drop_duplicates(subset=["osm_type", "osm_id"])
    .reset_index(drop=True)
)

print("Shape after removing duplicates:", stores_df.shape)

Shape after removing duplicates: (3781, 14)


In [32]:
# Assign a standard store group
import re

def identify_store_group(row):

    combined_text = " ".join([
        str(row["name"]),
        str(row["brand"]),
        str(row["operator"])
    ])

    if re.search(
        r"\bmigrolino\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "Migrolino"

    elif re.search(
        r"\bvoi\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "VOI Migros Partner"

    elif re.search(
        r"\bmigros\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "Migros"

    elif re.search(
        r"\bcoop\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "Coop"

    elif re.search(
        r"\bdenner\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "Denner"

    elif re.search(
        r"\blidl\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "Lidl"

    elif re.search(
        r"\baldi\b",
        combined_text,
        flags=re.IGNORECASE
    ):
        return "Aldi"

    else:
        return "Other"


stores_df["store_group"] = stores_df.apply(
    identify_store_group,
    axis=1
)

In [33]:
stores_df["store_group"].value_counts(dropna=False)

store_group
Coop                  1311
Denner                 810
Migros                 809
Migrolino              273
Aldi                   240
Lidl                   193
VOI Migros Partner      78
Other                   67
Name: count, dtype: int64

In [34]:
# Inspect false matches
false_matches_df = stores_df[
    stores_df["store_group"] == "Other"
]

print(
    "Possible false matches:",
    len(false_matches_df)
)

false_matches_df[
    [
        "name",
        "brand",
        "operator",
        "shop_type",
        "city"
    ]
].head(30)

Possible false matches: 67


,name,brand,operator,shop_type,city
654,Salon lavoir Montbrillant,NaN,NaN,laundry,Genève
781,Le Savoie,NaN,NaN,kiosk,NaN
910,Picaldi Jeans,NaN,NaN,clothes,Winterthur
940,Boulangerie Duvoisin,NaN,NaN,bakery,Servion
975,Baldinger Optik,NaN,NaN,optician,NaN
1015,Coopérative des Halles,NaN,NaN,convenience,Neuchâtel
1078,Savon-Lavoir,NaN,NaN,laundry,Genève
1127,Baldinger Optik,NaN,NaN,optician,NaN
1162,Salon Lavoir,NaN,NaN,laundry,Les Acacias
1230,Voit Sport,NaN,NaN,sports,Zürich


In [35]:
# Compare store groups and shop types
shop_type_table = pd.crosstab(
    stores_df["store_group"],
    stores_df["shop_type"],
    margins=True
)

shop_type_table

shop_type,art,bakery,beverages,bicycle,boat,books,boutique,car,car_repair,charity,...,supermarket,supermarket;convenience,tailor,ticket,travel_agency,vacant,variety_store,wine,yes,All
store_group,,,,,,,,,,,,,,,,,,,,,
Aldi,0,0,0,0,0,0,0,0,0,0,...,240,0,0,0,0,0,0,0,0,240
Coop,0,0,1,0,0,0,0,0,0,0,...,923,1,0,0,1,1,0,1,3,1311
Denner,0,0,2,0,0,0,0,0,0,0,...,724,0,0,0,0,0,0,0,0,810
Lidl,0,0,0,0,0,0,0,0,0,0,...,191,0,0,0,0,0,0,0,0,193
Migrolino,0,0,0,0,0,0,0,0,0,0,...,8,0,0,0,0,0,0,0,0,273
Migros,0,0,0,0,0,0,0,0,0,0,...,717,0,0,0,0,0,1,0,0,809
Other,1,2,0,2,2,1,1,1,3,1,...,3,0,2,1,1,0,0,0,1,67
VOI Migros Partner,0,0,0,0,0,0,0,0,0,0,...,74,0,0,0,0,0,0,0,0,78
All,1,2,3,2,2,1,1,1,3,1,...,2880,1,2,1,2,1,1,1,4,3781


In [36]:
shop_type_summary = (
    stores_df
    .groupby(["store_group", "shop_type"])
    .size()
    .reset_index(name="number_of_locations")
    .sort_values(
        ["store_group", "number_of_locations"],
        ascending=[True, False]
    )
)

shop_type_summary

,store_group,shop_type,number_of_locations
0,Aldi,supermarket,240
12,Coop,supermarket,923
2,Coop,convenience,329
3,Coop,department_store,22
5,Coop,florist,8
...,...,...,...
72,Other,travel_agency,1
73,Other,yes,1
76,VOI Migros Partner,supermarket,74
74,VOI Migros Partner,convenience,3


In [37]:
# keep only supermarket branches from the five relevant groups
supermarket_brands = [
    "Migros",
    "Coop",
    "Aldi",
    "Lidl",
    "Denner"
]

supermarkets_df = stores_df[
    (stores_df["shop_type"] == "supermarket")
    & (
        stores_df["store_group"].isin(
            supermarket_brands
        )
    )
].copy()

supermarkets_df.reset_index(
    drop=True,
    inplace=True
)

print("Supermarket dataset shape:", supermarkets_df.shape)

Supermarket dataset shape: (2795, 15)


In [38]:
supermarkets_df["shop_type"].value_counts(dropna=False)

shop_type
supermarket    2795
Name: count, dtype: int64

In [39]:
# check missing coordinates
print("Missing latitude:", supermarkets_df["latitude"].isna().sum())
print("Missing longitude:", supermarkets_df["longitude"].isna().sum())

Missing latitude: 0
Missing longitude: 0


In [40]:
# Remove records that cannot be mapped

supermarkets_df = (
    supermarkets_df
    .dropna(subset=["latitude", "longitude"])
    .reset_index(drop=True)
)

print("Final shape:", stores_df.shape)

Final shape: (3781, 15)


In [41]:
# Save the cleaned CSV
supermarkets_df.to_csv(
    "data/switzerland_supermarkets_clean.csv",
    index=False
)

print("Cleaned store data saved")

Cleaned store data saved


In [42]:
# verify the result
summary_df = (
    supermarkets_df["store_group"]
    .value_counts()
    .rename_axis("store_group")
    .reset_index(name="number_of_locations")
)

summary_df

,store_group,number_of_locations
0,Coop,923
1,Denner,724
2,Migros,717
3,Aldi,240
4,Lidl,191


In [43]:
supermarkets_df.head()

,osm_type,osm_id,name,brand,operator,shop_type,street,house_number,postal_code,city,latitude,longitude,opening_hours,website,store_group
0,node,36726161,Migros,Migros,Genossenschaft Migros Zürich,supermarket,Wiesentalstrasse,4,8730,Uznach,47.228183,8.965330,Mo-Th 07:30-19:00; Fr 07:30-20:00; Sa 07:30-17...,https://filialen.migros.ch/de/migros-supermark...,Migros
1,node,39768209,Coop,Coop,Coop Genossenschaft,supermarket,NaN,NaN,8730,Uznach,47.225154,8.969868,NaN,NaN,Coop
2,node,39947904,Coop,Coop,Coop Genossenschaft,supermarket,Bahnhofbrücke,1,8001,Zürich,47.376732,8.542161,Mo-Sa 06:00-22:00,NaN,Coop
3,node,48932835,Migros,Migros,Genossenschaft Migros Zürich,supermarket,Wengistrasse,7,8004,Zürich,47.375020,8.522895,Mo-Sa 08:00-21:00; PH off,https://filialen.migros.ch/de/migros-supermark...,Migros
4,node,70656488,Migros,Migros,Genossenschaft Migros Ostschweiz,supermarket,Zürcherstrasse,102,8406,Winterthur,47.491874,8.706448,Mo-Fr 07:30-20:00; PH off; Sa 08:00-19:00,https://filialen.migros.ch/de/migros-supermark...,Migros


In [44]:
supermarkets_df.columns

Index(['osm_type', 'osm_id', 'name', 'brand', 'operator', 'shop_type',
       'street', 'house_number', 'postal_code', 'city', 'latitude',
       'longitude', 'opening_hours', 'website', 'store_group'],
      dtype='str')

In [45]:
print(supermarkets_df["shop_type"].unique())
print(supermarkets_df["store_group"].unique())
print(supermarkets_df.shape)

<StringArray>
['supermarket']
Length: 1, dtype: str
<StringArray>
['Migros', 'Coop', 'Aldi', 'Denner', 'Lidl']
Length: 5, dtype: str
(2795, 15)


## Data source

Store-location data was collected from OpenStreetMap through the
Overpass API.

© OpenStreetMap contributors

The cleaned dataset was saved as:
`data/switzerland_supermarkets_clean.csv`